In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 17. Transformer Architecture — Assembling the Blocks

> **Learning Goals**
> - Understand the architecture of the original "Attention is All You Need" paper block by block
> - Explain the difference between Pre-LN and Post-LN and why it affects training stability
> - Compare encoder-decoder, encoder-only, and decoder-only uses

## 17.1 The Birth of the Transformer

Vaswani et al. (2017) showed that sequences can be processed with attention alone, without RNNs.

Benefits:
1. **Parallelization**: removes the sequential processing bottleneck of RNNs
2. **Long-range dependencies**: every token can directly attend to every other token
3. **Scalability**: model size is easier to increase, forming the basis of the LLM era

## 17.2 Three Variants

| Variant | Structure | Example Models |
|---|---|---|
| Encoder-decoder | both sides | machine translation, original Transformer, T5, BART |
| Encoder-only | bidirectional attention | BERT, RoBERTa |
| Decoder-only | causal attention | GPT, LLaMA, Qwen |

The LLM era is dominated by **decoder-only** models. Next-token prediction alone can learn powerful capabilities.

## 17.3 Transformer Block Structure

Each block contains:
1. Multi-Head Self-Attention
2. Residual connection
3. Normalization (LayerNorm/RMSNorm)
4. Feed-Forward Network (FFN)
5. Residual connection
6. Normalization

**Post-LN** (original):
$$\mathbf{x}' = \mathrm{LayerNorm}(\mathbf{x} + \mathrm{Attn}(\mathbf{x}))$$

**Pre-LN** (standard after GPT-2):
$$\mathbf{x}' = \mathbf{x} + \mathrm{Attn}(\mathrm{LayerNorm}(\mathbf{x}))$$

Pre-LN trains more stably in deep models.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

class TransformerBlock(nn.Module):
    """Pre-LN Transformer Block."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, pre_ln=True):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.pre_ln = pre_ln

    def attention(self, x, mask=None):
        B, T, D = x.shape
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask=None):
        if self.pre_ln:
            # Pre-LN
            x = x + self.dropout(self.attention(self.ln1(x), mask))
            x = x + self.dropout(self.ffn(self.ln2(x)))
        else:
            # Post-LN
            x = self.ln1(x + self.dropout(self.attention(x, mask)))
            x = self.ln2(x + self.dropout(self.ffn(x)))
        return x

# Test
d_model, n_heads, d_ff = 64, 8, 256
block = TransformerBlock(d_model, n_heads, d_ff)
x = torch.randn(2, 10, d_model)
out = block(x)
print(f"Input: {x.shape}")
print(f"Output: {out.shape}")
print(f"Block Parameter Count: {sum(p.numel() for p in block.parameters()):,}")


## 17.4 Position-wise Feed-Forward Network (FFN)

$$\mathrm{FFN}(\mathbf{x}) = \mathrm{GELU}(\mathbf{x} W_1 + \mathbf{b}_1) W_2 + \mathbf{b}_2$$

- $W_1 \in \mathbb{R}^{d \times d_{ff}}$, $W_2 \in \mathbb{R}^{d_{ff} \times d}$
- Usually $d_{ff} = 4d$ (for example, $d=512, d_{ff}=2048$)
- Applied **independently** at each position (position-wise)

**SwiGLU** (LLaMA): improves performance with a GLU-style activation
$$\mathrm{SwiGLU}(\mathbf{x}) = \mathrm{SiLU}(\mathbf{x} W_1) \odot (\mathbf{x} W_3) W_2$$


In [ ]:
# FFN variant comparison
class StandardFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff)
        self.w2 = nn.Linear(d_ff, d)

    def forward(self, x):
        return self.w2(F.gelu(self.w1(x)))

class SwiGLUFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d, bias=False)
        self.w3 = nn.Linear(d, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

d, d_ff = 64, 256
std_ffn = StandardFFN(d, d_ff)
glu_ffn = SwiGLUFFN(d, d_ff)
print(f"Standard FFN params: {sum(p.numel() for p in std_ffn.parameters()):,}")
print(f"SwiGLU FFN params: {sum(p.numel() for p in glu_ffn.parameters()):,}")

x = torch.randn(2, 10, d)
y1 = std_ffn(x)
y2 = glu_ffn(x)
print(f"Standard FFN Output: {y1.shape}")
print(f"SwiGLU FFN Output: {y2.shape}")
print("\n=> SwiGLU adds W3, so it has more parameters, but improves performance. It is the LLaMA standard.")


## 17.5 Role of Residual Connections and Normalization

**Residual connection**: $\mathbf{x}_{\ell+1} = \mathbf{x}_\ell + f_\ell(\mathbf{x}_\ell)$

- Provides a direct gradient path, making deep model training possible
- Prevents information bottlenecks

**Normalization**: stabilizes activation distributions, enabling larger learning rates and faster convergence.


In [ ]:
# Residual connection effect: gradient flow by depth
def test_grad_flow(depth, use_residual):
    """Simulate gradient flow by depth."""
    d = 64
    layers = [nn.Linear(d, d) for _ in range(depth)]
    x = torch.randn(1, d, requires_grad=True)
    target = torch.randn(1, d)
    loss_fn = nn.MSELoss()

    out = x
    for layer in layers:
        if use_residual:
            out = out + layer(out)
        else:
            out = layer(out)
    loss = loss_fn(out, target)
    loss.backward()
    return x.grad.norm().item()

print(f"{'depth':>6} | {'residual X':>15} | {'residual O':>15}")
print("-" * 40)
for d in [5, 10, 20, 50]:
    g_no = test_grad_flow(d, False)
    g_yes = test_grad_flow(d, True)
    print(f"{d:>6} | {g_no:>15.6e} | {g_yes:>15.6e}")
print("\n=> Residual connections keep gradients flowing even as depth increases.")


## 17.6 Counting Model Parameters

Per transformer block:
- QKV projections: $3 d^2$
- Output projection: $d^2$
- FFN (standard): $2 \cdot 4d^2 = 8d^2$
- LayerNorm: $4d$
- **Total**: about $12 d^2$ per block

Whole model ($L$ blocks):
- Embedding: $|V| \cdot d$
- Blocks: $L \cdot 12 d^2$
- Output layer: $|V| \cdot d$ (omitted when using weight tying)

Roughly $P \approx 12 L d^2$ when embeddings are ignored.


In [ ]:
# parameter count calculation
def count_transformer_params(vocab_size, d_model, n_layers, d_ff=None, tie_weights=True):
    if d_ff is None: d_ff = 4 * d_model
    # Embedding
    emb = vocab_size * d_model
    # Per block
    qkv = 3 * d_model * d_model
    out_proj = d_model * d_model
    ffn = 2 * d_model * d_ff
    ln = 4 * d_model  # two LNs
    block = qkv + out_proj + ffn + ln
    # outputlayer
    output = 0 if tie_weights else vocab_size * d_model
    total = emb + n_layers * block + output
    return total, {'emb': emb, 'block': block, 'n_layers': n_layers, 'output': output}

# Scale comparison
print(f"{'Model':>12} | {'V':>8} | {'d':>5} | {'L':>3} | {'Params':>12}")
print("-" * 50)
for name, V, d, L in [
    ('Mini', 1000, 64, 4),
    ('Small', 10000, 256, 6),
    ('GPT-1', 40000, 768, 12),
    ('GPT-2', 50257, 768, 12),
    ('GPT-2 medium', 50257, 1024, 24),
    ('GPT-3 6.7B', 50257, 4096, 32),
    ('LLaMA-7B', 32000, 4096, 32),
]:
    n, _ = count_transformer_params(V, d, L)
    print(f"{name:>12} | {V:>8} | {d:>5} | {L:>3} | {n:>12,}")


## 17.7 Key Takeaways

| Component | Role |
|---|---|
| Multi-Head Attention | learns relationships between tokens |
| FFN | nonlinear position-wise transformation |
| Residual connection | enables deep model training |
| LayerNorm/RMSNorm | stabilizes activations |
| Positional Encoding | injects order information |

| Variant | Use |
|---|---|
| Post-LN | original Transformer |
| Pre-LN | GPT-2+ standard |
| SwiGLU FFN | LLaMA standard |

## Exercises

1. Train Pre-LN and Post-LN blocks at the same depth and compare stability.
2. Compare Standard FFN and SwiGLU FFN with the same parameter count (adjust $d_{ff}$ as needed).
3. Train models with and without residual connections at depths 10, 20, and 50, then compare gradients.
4. Compute the parameter count of GPT-2 small ($d=768, L=12, V=50257$).
5. Compute the parameter ratio between attention and FFN inside a transformer block.

> Solutions: `solutions/ch17_solutions.ipynb`
